In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("/Users/biman_giri/Documents/OfficeWork/MyDay2DayLearning/.env")
openai_token = os.getenv("OPENAI_API_TOKEN")
openai_base_url = os.getenv("OPENAI_API_BASE")
lanchain_endpoint = os.getenv("LANGSMITH_ENDPOINT")
lanchain_api_key = os.getenv("LANGSMITH_API_KEY")
lanchain_project = os.getenv("LANGSMITH_PROJECT")
lanchain_tracing_v2 = os.getenv("LANGSMITH_TRACING")

In [8]:
from langchain_openai import ChatOpenAI

llm_model = ChatOpenAI(model="gpt-4o", api_key=openai_token, base_url=openai_base_url)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

# message = [SystemMessage(content = "help me to translate the following text from English to {language}"), HumanMessage(content = "{text}")]
message = [
    ("system", "help me to translate the following text from English to {language}"),
    ("user", "{text}"),
]
prompt = ChatPromptTemplate.from_messages(message)
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
chain = prompt | llm_model | parser
chain.invoke({"text": "I love programming in Python", "language": "Bengali"})

'আমি পাইথনে প্রোগ্রামিং করতে ভালোবাসি।'

In [10]:
llm_model.invoke([HumanMessage(content="I love programming in Python")])

AIMessage(content="That's great to hear! Python is a versatile, high-level programming language that is widely used for a variety of applications, including web development, data analysis, machine learning, and automation. Its simple and readable syntax makes it an excellent choice for both beginners and experienced programmers. If you have any specific questions or topics you'd like to explore further in Python, feel free to ask!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 12, 'total_tokens': 87, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_c7a156cce7', 'id': 'chatcmpl-DIeK1fwYBKszq8YLkcpB57j4gIm1y', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}

In [13]:
from langchain_core.messages import AIMessage

resonse = llm_model.invoke(
    [
        HumanMessage(content="I love programing in python"),
        AIMessage(
            content="That's great to hear! Python is a versatile, high-level programming language that is widely used for a variety of applications, including web development, data analysis, machine learning, and automation. Its simple and readable syntax makes it an excellent choice for both beginners and experienced programmers. If you have any specific questions or topics you'd like to explore further in Python, feel free to ask!"
        ),
        HumanMessage(content="which language i love for programming?"),
    ]
)
print(resonse.content)

It sounds like you love programming in Python! If you're open to exploring other languages or have specific interests (like web development, data science, or systems programming), I can suggest some additional languages that might align with those areas. Let me know if you'd like more information!


#### Message history
we can use the message history to wrap our model and make the model stateful. This will keep track of input and output of the model , and store them in some datastore. Futrue interactio will load those messages and pass them into the chain as part of the input.


In [14]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = dict()


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    return store[session_id]


with_message_history = RunnableWithMessageHistory(llm_model, get_session_history)

config = {"configurable": {"session_id": "chat1"}}
with_message_history.invoke(
    input=[
        HumanMessage(content="I love programing in python"),
    ],
    config=config,
)

AIMessage(content="That's great to hear! Python is a versatile and powerful programming language that's popular for its readability and simplicity. Whether you're working on web development, data analysis, artificial intelligence, or automation, Python has a wide range of libraries and frameworks to support your projects. If you have any questions or need tips on Python programming, feel free to ask!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 13, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_c7a156cce7', 'id': 'chatcmpl-DIeedOsZk2wRSa0RV6LtEEHzdu4S1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ce336-e02a-794

In [15]:
# with the same session id, the model will remember the previous message
with_message_history.invoke(
    input=[
        HumanMessage(content="which programming language i love for programming?"),
    ],
    config=config,
)

AIMessage(content="It seems like you already have a fondness for Python based on your earlier statement. However, if you’re considering exploring other programming languages, your preferences might depend on your specific interests, projects, and goals. Here are a few popular programming languages, along with common use cases, to help you decide:\n\n1. **JavaScript**: Great for web development. It’s essential for front-end development and also used on the server-side with Node.js.\n\n2. **Java**: Known for its portability across platforms, it's widely used in enterprise environments, Android app development, and large system architectures.\n\n3. **C++**: Ideal for systems programming, game development, and performance-critical applications. It provides a lot of control over system resources.\n\n4. **C#**: Used extensively for developing Windows applications and games using the Unity game engine.\n\n5. **Ruby**: Known for its elegant and concise syntax, it's popular for web development,

In [16]:
# lets try with different session id
config = {"configurable": {"session_id": "chat2"}}
with_message_history.invoke(
    input=[
        HumanMessage(content="which programming language i love for programming?"),
    ],
    config=config,
)

AIMessage(content="I can't determine your personal preferences unless you provide more context. However, if you're looking for popular languages to explore, here are a few options based on common preferences and strengths:\n\n1. **Python**: Known for its readability and versatility, great for beginners and experienced developers alike.\n\n2. **JavaScript**: Essential for web development, as it runs in browsers and on servers with Node.js.\n\n3. **Java**: Used for Android development and large-scale enterprise applications.\n\n4. **C++**: Preferred for systems programming, game development, and scenarios requiring high-performance software.\n\n5. **Ruby**: Loved for its elegant syntax and is commonly used with the Ruby on Rails framework.\n\n6. **Rust**: Appreciated for its focus on safety and performance, often used in systems programming.\n\n7. **Go**: Known for its simplicity and efficiency, ideal for cloud services and backend development.\n\nThink about what you want to achieve wit

In [17]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant, answer the question based on your ability",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
chain = prompt | llm_model
chain.invoke({"messages": [HumanMessage(content="Hi, My name is Biman")]})

AIMessage(content='Hello Biman! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 31, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_c7a156cce7', 'id': 'chatcmpl-DIereBWrNEhvi1cmQQVIyGbjfTb3s', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ce343-31a9-7011-9996-9aa09a10fefa-0', usage_metadata={'input_tokens': 31, 'output_tokens': 11, 'total_tokens': 42, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [18]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)
config = {"configurable": {"session_id": "chat3"}}
with_message_history.invoke(
    input=[
        HumanMessage(content="Hi, My name is Biman"),
    ],
    config=config,
)

AIMessage(content='Hello Biman! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 31, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a2a2b96670', 'id': 'chatcmpl-DIetvNv0YF34kJIBekazFsQmK8Luv', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ce345-5805-71b0-8df7-50b54f1fe541-0', usage_metadata={'input_tokens': 31, 'output_tokens': 11, 'total_tokens': 42, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [20]:
### add more complexity
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant, answer the question based on your ability in {language}",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
chain = prompt | llm_model
response = chain.invoke(
    {"messages": [HumanMessage(content="Hi, My name is Biman")], "language": "Bengali"}
)
response.content

'হ্যালো বিমল! কেমন আছেন? কীভাবে আমি আপনার সাহায্য করতে পারি?'

In [21]:
with_message_history = RunnableWithMessageHistory(
    chain, get_session_history, input_messages_key="messages"
)
config = {"configurable": {"session_id": "chat4"}}
response = with_message_history.invoke(
    input={
        "messages": [
            HumanMessage(
                content="Hi, my name is Biman",
            )
        ],
        "language": "Hindi",
    },
    config=config,
)
response.content

'नमस्ते, बिमान! कैसे हैं आप? आज मैं आपकी किस प्रकार से मदद कर सकता हूँ?'

In [22]:
with_message_history.invoke(
    input={
        "messages": [
            HumanMessage(
                content="what is my name?",
            )
        ],
        "language": "Hindi",
    },
    config=config,
)
response.content

'नमस्ते, बिमान! कैसे हैं आप? आज मैं आपकी किस प्रकार से मदद कर सकता हूँ?'

### Prompt Template
Prompt templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom interuciton (but still taking message as input). Next, we will add in more input besides just the messsage.

In [23]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are helpful assistant, answer the quesition to the best of your knowledge",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
prompt.invoke({"messages": [HumanMessage(content="what is my name?")]})

ChatPromptValue(messages=[SystemMessage(content='You are helpful assistant, answer the quesition to the best of your knowledge', additional_kwargs={}, response_metadata={}), HumanMessage(content='what is my name?', additional_kwargs={}, response_metadata={})])

In [25]:
chain = prompt | llm_model | StrOutputParser()
store = dict()
from langchain_core.chat_history import InMemoryChatMessageHistory


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


with_message_history = RunnableWithMessageHistory(chain, get_session_history)
config = {"configurable": {"session_id": "chat6"}}
response = with_message_history.invoke(
    input={"messages": [HumanMessage(content="Tell me about myself")]},
    config=config,
)
print("first response", response)
response = with_message_history.invoke(
    input={
        "messages": [
            HumanMessage(
                content="Hello my name is Biman, I am Generative AI Enginne, love to work in python"
            )
        ]
    },
    config=config,
)

print("second response", response)
response = with_message_history.invoke(
    input={"messages": [HumanMessage(content="Tell me about myself")]},
    config=config,
)
print("third response", response)

first response I'm sorry, but I don't have access to personal data about individuals unless it has been shared with me in the course of our conversation. I can help answer questions or provide information based on the details you choose to share. How can I assist you today?
second response Hello Biman! It's great to meet you. As a Generative AI Engineer with a passion for Python, you’re likely involved in some cutting-edge projects. Python is a powerful language for AI development due to its extensive libraries and frameworks such as TensorFlow, PyTorch, and Keras. If you have any specific questions about Python, AI, or anything else, feel free to ask!
third response While I don't have personal knowledge beyond what you've shared, based on the information you've provided, here's what I can summarize about you:

Your name is Biman, and you are a Generative AI Engineer. You have a strong interest in Python, which indicates you likely have expertise in programming and enjoy working with m

In [49]:
chain = prompt | llm_model | StrOutputParser()
store = dict()
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are language translator, translate the following text from {source_language} to {target_language}. ",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
chain = prompt | llm_model | StrOutputParser()
chain.invoke(
    {
        "messages": [
            HumanMessage(
                content="My name is Biman, I love to programming in python, I am a generative AI engineer in one the leading consulting firm in MBB. "
            )
        ],
        "source_language": "English",
        "target_language": "Bengali",
    }
)

'আমার নাম বিমল, আমি পাইথনে প্রোগ্রামিং করতে ভালোবাসি, আমি এমবিবি-র অন্যতম প্রধান পরামর্শক প্রতিষ্ঠানের একটি প্রজন্মীয় এআই ইঞ্জিনিয়ার।'

In [50]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


chain = prompt | llm_model | StrOutputParser()
with_message_history = RunnableWithMessageHistory(
    chain, get_session_history, input_messages_key="messages"
)
config = {"configurable": {"session_id": "chat9"}}
response = with_message_history.invoke(
    input={
        "messages": [
            HumanMessage(
                content="My name is Biman, I love to programming in python, I am a generative AI engineer in one the leading consulting firm in MBB. "
            )
        ],
        "source_language": "English",
        "target_language": "Bengali",
    },
    config=config,
)
print("first response", response)

first response আমার নাম বিমল, আমি পাইথনে প্রোগ্রামিং করতে ভালোবাসি, আমি এমবিবির অন্যতম শীর্ষস্থানীয় কনসালটিং ফার্মে একজন জেনারেটিভ এআই ইঞ্জিনিয়ার।


In [53]:
response = with_message_history.invoke(
    input={
        "messages": [HumanMessage(content="dont translate it, now what was my name? ")],
        "source_language": "English",
        "target_language": "Odia",
    },
    config=config,
)
print("second response", response)

second response Your name was Biman.


In [55]:
with_message_history.get_session_history(config["configurable"]["session_id"])

InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is Biman, I love to programming in python, I am a generative AI engineer in one the leading consulting firm in MBB. ', additional_kwargs={}, response_metadata={}), AIMessage(content='আমার নাম বিমল, আমি পাইথনে প্রোগ্রামিং করতে ভালোবাসি, আমি এমবিবির অন্যতম শীর্ষস্থানীয় কনসালটিং ফার্মে একজন জেনারেটিভ এআই ইঞ্জিনিয়ার।', additional_kwargs={}, response_metadata={}), HumanMessage(content='what is my name?', additional_kwargs={}, response_metadata={}), AIMessage(content='ତୁମର ନାମ ବିମାନ।', additional_kwargs={}, response_metadata={}), HumanMessage(content='dont translate it, now what was my name? ', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name was Biman.', additional_kwargs={}, response_metadata={})])

#### Managing the conversational history
One important concept to understand when building chatbots is how to manage conversation history. If left unmanged, the list of messages will gorw unbounded and potentially overlfow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.


In [89]:
from langchain_core.messages import SystemMessage, trim_messages

### trim message : helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages.


messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="Hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice creatm"),
    AIMessage(content="nice, I like it too"),
    HumanMessage(content="I'm going to the movies later"),
    AIMessage(content="have fun!"),
    HumanMessage(content="My name is Biman giri"),
    AIMessage(content="your name is bob"),
]

In [92]:
trimmer = trim_messages(
    max_tokens=40,
    strategy="last",
    token_counter=llm_model,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='My name is Biman giri', additional_kwargs={}, response_metadata={}),
 AIMessage(content='your name is bob', additional_kwargs={}, response_metadata={})]

In [93]:
trimmer = trim_messages(
    max_tokens=50,
    strategy="last",
    token_counter=llm_model,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="I'm going to the movies later", additional_kwargs={}, response_metadata={}),
 AIMessage(content='have fun!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='My name is Biman giri', additional_kwargs={}, response_metadata={}),
 AIMessage(content='your name is bob', additional_kwargs={}, response_metadata={})]

In [96]:
trimmer = trim_messages(
    max_tokens=100,
    strategy="last",
    token_counter=llm_model,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="Hi! I'm bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice creatm', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice, I like it too', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="I'm going to the movies later", additional_kwargs={}, response_metadata={}),
 AIMessage(content='have fun!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='My name is Biman giri', additional_kwargs={}, response_metadata={}),
 AIMessage(content='your name is bob', additional_kwargs={}, response_metadata={})]

quick observation : some observation that I found out that as we make the include_system = true , so it always consider the SystemMessage, and then because we have make the start_on = "human" , it alwasy start with human message after the system message.

In [97]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)
    | prompt
    | llm_model
    | StrOutputParser()
)


chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what is my name?")],
        "source_language": "English",
        "target_language": "Bengali",
    }
)

'Your name is Biman Giri.'

In [105]:
# lets wrap this in the message history
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

trimmer = trim_messages(
    max_tokens=1000,
    strategy="last",
    token_counter=llm_model,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

store = dict()

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful ai assistant , who can asnwer all the question which can be asked",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)
    | prompt
    | llm_model
    | StrOutputParser()
)

with_message_history = RunnableWithMessageHistory(
    chain, get_session_history, input_messages_key="messages"
)

config = {"configurable": {"session_id": "chat11"}}
response = with_message_history.invoke(
    input={"messages": [HumanMessage(content="I am from india.")]},
    config=config,
)
print(response)

That's great! India is a diverse and culturally rich country with a rich history and a wide variety of languages, traditions, and cuisines. If you have any questions or need information on a particular topic related to India or anything else, feel free to ask!


In [106]:
response = with_message_history.invoke(
    input={"messages": [HumanMessage(content="how is mckinsey as a consulting firm ? I am going to apply for the AI enginner role. ")]},
    config=config,
)
print(response)

McKinsey & Company is one of the most prestigious consulting firms in the world, known for its work with businesses, governments, and institutions across various industries. Here are some key points about McKinsey, particularly in relation to applying for an AI Engineer role:

1. **Reputation and Prestige**: McKinsey is highly regarded for its strategic acumen and thought leadership in consulting. It employs top talent from around the world and is known for offering insightful solutions and fostering innovation.

2. **Global Reach**: With offices in numerous countries, McKinsey serves a global clientele. This global presence provides opportunities for working on diverse projects that have international implications.

3. **Strong Focus on Technology and Digital**: In recent years, McKinsey has invested significantly in its technology, data science, and digital capabilities. The firm helps clients leverage AI, machine learning, and data analytics to create more efficient operations and i

In [107]:
with_message_history.get_session_history(config["configurable"]["session_id"]).messages

[HumanMessage(content='I am from india.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's great! India is a diverse and culturally rich country with a rich history and a wide variety of languages, traditions, and cuisines. If you have any questions or need information on a particular topic related to India or anything else, feel free to ask!", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='how is mckinsey as a consulting firm ? I am going to apply for the AI enginner role. ', additional_kwargs={}, response_metadata={}),
 AIMessage(content="McKinsey & Company is one of the most prestigious consulting firms in the world, known for its work with businesses, governments, and institutions across various industries. Here are some key points about McKinsey, particularly in relation to applying for an AI Engineer role:\n\n1. **Reputation and Prestige**: McKinsey is highly regarded for its strategic acumen and thought leadership in consulting. It emp

In [110]:
trimmer.invoke(with_message_history.get_session_history(config["configurable"]["session_id"]).messages) 

[HumanMessage(content='I am from india.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's great! India is a diverse and culturally rich country with a rich history and a wide variety of languages, traditions, and cuisines. If you have any questions or need information on a particular topic related to India or anything else, feel free to ask!", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='how is mckinsey as a consulting firm ? I am going to apply for the AI enginner role. ', additional_kwargs={}, response_metadata={}),
 AIMessage(content="McKinsey & Company is one of the most prestigious consulting firms in the world, known for its work with businesses, governments, and institutions across various industries. Here are some key points about McKinsey, particularly in relation to applying for an AI Engineer role:\n\n1. **Reputation and Prestige**: McKinsey is highly regarded for its strategic acumen and thought leadership in consulting. It emp

In [109]:
response = with_message_history.invoke(
    input={"messages": [HumanMessage(content="what is my president & prime minister name? and what is my experties? ")]},
    config=config,
)
print(response)

As of the latest information available, the President of India is **Droupadi Murmu**, who took office on July 25, 2022. The Prime Minister of India is **Narendra Modi**, who has been in office since May 26, 2014.

Regarding your expertise, I don't have access to personal information about individuals unless it has been shared in our interaction. If you tell me more about your skills, education, or professional background, I can certainly help you better understand or explore areas related to your expertise!


In [111]:
response = with_message_history.invoke(
    input={"messages": [HumanMessage(content="I alreay told that I am going to apply for the AI enginner role. ")]},
    config=config,
)
print(response)

Thank you for the reminder! As you're applying for an AI Engineer role, your expertise likely includes areas such as artificial intelligence, machine learning, data analysis, and possibly programming languages and tools commonly used in AI development (such as Python, TensorFlow, PyTorch, etc.).

AI engineers are responsible for developing models and algorithms that enable machines to perform tasks that typically require human intelligence. Your work might involve designing machine learning models, implementing AI solutions, and collaborating with data scientists and software engineers to integrate AI capabilities into products or services.

If you have specific questions about the AI field or need tips on preparing for the application process, including crafting your resume or preparing for interviews, feel free to ask!
